# HyDE Transformer
> Expand queries with hypothetical document embeddings for better retrieval.

`HyDETransformer` generates a hypothetical document that *would* answer the
query, then uses that document's embedding for retrieval instead of the
original query. This bridges the vocabulary gap between short queries and
long documents.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.query import HyDETransformer
from anchor.models import QueryBundle

## Define a Generation Function
HyDE needs a function that takes a query string and returns a hypothetical
document. In production this would call an LLM; here we use a mock.

In [ ]:
def mock_generate(query: str) -> str:
    """Simulate an LLM generating a hypothetical answer document."""
    return (
        f"A hypothetical document about {query} would discuss the fundamentals "
        f"of the topic, covering key concepts, common patterns, and practical "
        f"applications. It would explain how {query} relates to modern software "
        f"engineering practices and provide concrete examples."
    )

# Test the generation function
sample = mock_generate("context engineering")
print(f"Generated document ({len(sample)} chars):")
print(f"  {sample[:80]}...")

## Create the HyDE Transformer
Pass the generation function to `HyDETransformer`. The transformer will
call it on each query to produce a hypothetical document.

In [ ]:
hyde = HyDETransformer(generate_fn=mock_generate)

print(f"Transformer: {type(hyde).__name__}")
print(f"Generate fn: {hyde.generate_fn.__name__}")

## Transform a Query
Wrap the query string in a `QueryBundle`, then call `transform()`. The
result is a list of `QueryBundle` objects -- the original plus the
hypothetical document version.

In [ ]:
query = QueryBundle(query_str="What is context engineering?")
print(f"Original query: {query.query_str}")

expanded = hyde.transform(query)

print(f"\nExpanded to {len(expanded)} queries:\n")
for i, q in enumerate(expanded):
    print(f"  [{i}] ({len(q.query_str)} chars): {q.query_str[:70]}...")

## Try Different Queries
HyDE works best when queries are short and would benefit from expansion
into richer document-like text.

In [ ]:
test_queries = [
    "What is context engineering?",
    "How does RAG work?",
    "vector database comparison",
    "memory management in LLM applications",
]

for q_str in test_queries:
    q = QueryBundle(query_str=q_str)
    results = hyde.transform(q)
    hypo = results[-1].query_str  # last one is the hypothetical doc
    print(f"Query: {q_str}")
    print(f"  HyDE: {hypo[:65]}...")
    print()

## When to Use HyDE
HyDE shines when there is a vocabulary mismatch between user queries and
stored documents.

In [ ]:
# Compare query lengths before and after HyDE
queries = ["RAG", "How to build a chatbot", "Explain transformer attention"]

print(f"{'Query':<30} {'Original':>10} {'HyDE':>10} {'Expansion':>10}")
print("-" * 65)
for q_str in queries:
    q = QueryBundle(query_str=q_str)
    results = hyde.transform(q)
    orig_len = len(q_str)
    hyde_len = len(results[-1].query_str)
    ratio = hyde_len / orig_len
    print(f"{q_str:<30} {orig_len:>10} {hyde_len:>10} {ratio:>9.1f}x")

## Key Takeaways
- `HyDETransformer` generates a hypothetical answer document for each query
- The hypothetical document is used for embedding-based retrieval
- Bridges the vocabulary gap between short queries and long documents
- Requires a generation function (LLM call in production, mock for testing)
- Best for short, keyword-style queries that need expansion

**Next:** [Multi-Query Expansion](02_multi_query.ipynb)